In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install scikit-bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.7 MB/s eta 0:00:00


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist, squareform

from skbio.stats.distance import DistanceMatrix, permanova

from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

from sklearn.ensemble import IsolationForest

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score, r2_score, mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_excel("Data_vascular.xlsx")

In [ ]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,20,1.20,1.0,100,80,100,70,1
1,23,1.10,0.9,98,78,98,78,1
2,22,1.40,1.1,96,85,99,79,1
3,27,1.20,1.2,90,89,90,83,1
4,21,1.80,0.8,92,95,96,82,1
5,17,1.30,0.9,96,88,92,72,1
6,13,1.70,0.7,100,92,97,71,1
7,67,0.30,1.9,77,50,40,25,0
8,69,2.00,2.2,74,55,44,19,0
9,71,0.30,1.8,68,43,39,35,0


In [ ]:
# Shuffle original dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,71,0.30,1.8,68,43,39,35,0
1,68,0.34,1.9,75,47,35,34,0
2,20,1.20,1.0,100,80,100,70,1
3,79,0.19,1.8,70,43,41,28,0
4,17,1.30,0.9,96,88,92,72,1
5,69,2.00,2.2,74,55,44,19,0
6,22,1.40,1.1,96,85,99,79,1
7,23,1.10,0.9,98,78,98,78,1
8,70,0.35,2.1,69,59,46,22,0
9,21,1.80,0.8,92,95,96,82,1


In [ ]:
# Standardization
def standardization(x):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    return x_scaled, scaler

# Statistical Modelling

In [ ]:
def statistics(df):
    summary_dict = df.describe().to_dict()
    return json.dumps(summary_dict, indent=4)

In [ ]:
# Group Wise Descriptive Statistics for each feature
def group_wise_stats(df, label_col):
    if label_col not in df.columns:
      raise ValueError(f"{label_col} column not found in dataframe.")

    result = {}

    for col in df.columns[:-1]:
        stats = df.groupby(label_col)[col].describe().round(2)
        result[col] = stats.to_dict()

    return json.dumps(result, indent=4)

## t-test

In [ ]:
def t_test(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"{label_col} column not found in dataframe.")

    # Select numeric features except label
    features = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    # Get unique groups
    groups = df[label_col].unique()

    if len(groups) != 2:
        raise ValueError("Independent t-test requires exactly two groups.")

    g1_label, g0_label = groups

    results = {}

    for feature in features:

        # Extract actual data for each group
        group1 = df[df[label_col] == g1_label][feature].dropna()
        group0 = df[df[label_col] == g0_label][feature].dropna()

        if len(group1) < 2 or len(group0) < 2:
            continue

        t_stat, p_value = ttest_ind(group1, group0, equal_var=False)

        pooled_std = np.sqrt((group1.var() + group0.var()) / 2)

        cohens_d = 0.0 if pooled_std == 0 else \
            (group1.mean() - group0.mean()) / pooled_std

        results[feature] = {
            "group1_label": g1_label,
            "group0_label": g0_label,
            "mean_group1": float(round(group1.mean(), 3)),
            "mean_group0": float(round(group0.mean(), 3)),
            "t_statistic": float(round(t_stat, 3)),
            "p_value": float(round(p_value, 6)),
            "cohens_d": float(round(cohens_d, 3)),
            "significant": bool(p_value < 0.05)
        }

    return results

## Mann–Whitney U Test

In [ ]:
def mann_whitney_test(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    if df[label_col].nunique() != 2:
        raise ValueError("Mann–Whitney test requires exactly 2 groups.")

    results = {}

    # Numeric columns except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    for feature in numeric_cols:

        group_values = df[[feature, label_col]].dropna()

        groups = group_values[label_col].unique()
        g0 = group_values[group_values[label_col] == groups[0]][feature]
        g1 = group_values[group_values[label_col] == groups[1]][feature]

        if len(g0) < 1 or len(g1) < 1:
            continue

        try:
            u_stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')
        except Exception:
            continue

        results[feature] = {
            "median_group1": float(round(g0.median(), 3)),
            "median_group2": float(round(g1.median(), 3)),
            "U_statistic": float(round(u_stat, 3)),
            "p_value": float(round(p_val, 6)),
            "significant": bool(p_val < 0.05)
        }

    return results

## MANOVA Test

In [ ]:
def manova(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe.")

    if df[label_col].nunique() < 2:
        raise ValueError("MANOVA requires at least two groups.")

    # Select numeric features except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    if len(numeric_cols) < 1:
        raise ValueError("No numeric dependent variables found.")

    # Build formula dynamically
    dependent_vars = " + ".join(numeric_cols)
    formula = f"{dependent_vars} ~ {label_col}"

    try:
        manova = MANOVA.from_formula(formula, data=df)
        result = manova.mv_test()
    except Exception as e:
        raise RuntimeError(f"MANOVA failed: {str(e)}")

    output = {
        "method": "MANOVA",
        "effects": {}
    }

    for effect, effect_data in result.results.items():
        stat_table = effect_data["stat"]

        effect_dict = {}

        for test_name, row in stat_table.iterrows():

            # Normalize test name
            normalized_name = (
                test_name.lower()
                .replace(" ", "_")
                .replace("-", "_")
                .replace("'", "")
            )

            effect_dict[normalized_name] = {
                "Value": float(row["Value"]),
                "Num DF": float(row["Num DF"]),
                "Den DF": float(row["Den DF"]),
                "F Value": float(row["F Value"]),
                "Pr > F": float(row["Pr > F"])
            }

        output["effects"][effect] = effect_dict

    return output

## PERMANOVA Test

In [ ]:
def permanova_test(df, label_col, metric="euclidean", permutations=999):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    # group validation
    if df[label_col].nunique() < 2:
        raise ValueError("PERMANOVA requires at least two groups.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features found for PERMANOVA.")

    # missing value check
    cols = feature_cols + [label_col]
    if df[cols].isnull().any().any():
        raise ValueError("Missing values detected. Please clean the dataset.")

    # feature matrix
    X = df[feature_cols].values
    groups = df[label_col].values

    try:
        dist_matrix = squareform(pdist(X, metric=metric))
        dm = DistanceMatrix(dist_matrix)

        result = permanova(
            distance_matrix=dm,
            grouping=groups,
            permutations=permutations
        )

    except Exception as e:
        raise RuntimeError(f"PERMANOVA failed: {str(e)}")

    return {
        "method_name": "PERMANOVA",
        "metric": metric,
        "permutations": permutations,
        "test_statistic_name": "pseudo-F",
        "test_statistic": float(result["test statistic"]),
        "p_value": float(result["p-value"]),
        "sample_size": int(result["sample size"]),
        "number_of_groups": int(result["number of groups"]),
        "significant": bool(result["p-value"] < 0.05)
    }

## Correlation Heatmap

In [ ]:
def correlation_matrix(df, label_col=None, method="pearson"):

    if method not in ["pearson","spearman","kendall"]:
        raise ValueError("Method must be 'pearson', 'spearman', or 'kendall'.")

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # remove label column if numeric
    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) < 2:
        raise ValueError("At least two numeric features are required to compute correlation.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in dataset.")

    # compute correlation matrix
    corr_matrix = df[feature_cols].corr(method=method).round(3)

    return {
        "method_used": method,
        "features": feature_cols,
        "correlation_matrix": corr_matrix.to_dict()
    }

In [ ]:
def statistical_modelliing(df, test_name, label_col=None, **kwargs):
    if test_name == "df_summary":
        return statistics(df)

    elif test_name == "group_wise_summary":
        return group_wise_stats(df, label_col)

    elif test_name == "t_test":
        return t_test(df, label_col)

    elif test_name == "mann_whitney_test":
        return mann_whitney_test(df, label_col)

    elif test_name == "manova":
        return manova(df, label_col)

    elif test_name == "permanova":
        return permanova_test(df, label_col, metric=kwargs.get("metric", "euclidean"), permutations=kwargs.get("permutations", 999))

    elif test_name == "correlation_matrix":
        return correlation_matrix(df, label_col, method=kwargs.get("method", "pearson"))

    else:
        raise ValueError(f"Invalid test name provided: {test_name}")

# Anamoly Detection

# 1). Z-Score

In [ ]:
def zscore_outliers(df, label_col=None, threshold=3):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # select numeric features automatically
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Z-score analysis.")

    # check missing values
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    # feature matrix
    X = df[feature_cols]

    # compute Z-scores
    z_scores = np.abs(stats.zscore(X))
    anomaly_mask = (z_scores > threshold).any(axis=1)

    anomalies = df.loc[anomaly_mask].copy()

    return {
        "threshold": threshold,
        "total_rows": int(len(df)),
        "num_anomalies": int(anomaly_mask.sum()),
        "anomaly_rows": anomalies.to_dict(orient="records")
    }

## 2) IQR (Box Plot)

In [ ]:
def boxplot(df, label_col):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for boxplot.")

    # check missing values
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    results = {}

    for feature in feature_cols:
        results[feature] = {}

        for group, group_df in df.groupby(label_col):

            values = group_df[feature]

            q1 = np.percentile(values, 25)
            median = np.percentile(values, 50)
            q3 = np.percentile(values, 75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            # TRUE whiskers (closest value within bounds)
            lower_whisker = values[values >= lower_bound].min()
            upper_whisker = values[values <= upper_bound].max()

            # Outliers
            outlier_mask = (values < lower_bound) | (values > upper_bound)
            outlier_rows = group_df.loc[outlier_mask]

            results[feature][str(group)] = {
                "min": float(values.min()),
                "q1": float(q1),
                "median": float(median),
                "q3": float(q3),
                "max": float(values.max()),
                "lower_whisker": float(lower_whisker),
                "upper_whisker": float(upper_whisker),
                "outliers": values[outlier_mask].tolist(),
                "outlier_rows": outlier_rows.to_dict(orient="records"),
                "count": int(len(values))
            }

    return results

## 3) Isolation Forest

In [ ]:
def predict_isolation_forest(df):

    model_path = "/content/drive/MyDrive/rasp/models/isolation_forest_model.pkl"

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be pandas DataFrame")

    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    X = df[feature_cols]

    X_scaled = scaler.transform(X)

    predictions = model.predict(X_scaled)

    scores = model.decision_function(X_scaled)

    anomaly_mask = predictions == -1

    anomalies = df.loc[anomaly_mask]

    result = {
        "total_rows": int(len(df)),
        "num_anomalies": int(anomaly_mask.sum()),
        "anomaly_indices": anomalies.index.tolist(),
        "anomaly_rows": anomalies.to_dict(orient="records"),
        "scores": scores.tolist()
    }

    return result

In [ ]:
def ananomaly_detection(df, test_name, label_col=None, **kwargs):
    if test_name == "z_score":
        return zscore_outliers(df, label_col, threshold=kwargs.get("threshold", 3))

    elif test_name == "boxplot":
        return boxplot(df, label_col)

    elif test_name == "isolation_forest":
        return predict_isolation_forest(df)

    else:
        raise ValueError(f"Invalid test name: {test_name}")

# Predictive Modelling

## Classification

In [ ]:
# Validation
def validate_input(df, label_col):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col is not None and label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataset.")

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if df[label_col].nunique() != 2:
        raise ValueError("Binary classification requires exactly 2 classes.")

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Isolation Forest.")

    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    if len(df) < 10:
        raise ValueError("Dataset too small for modeling.")

### Multicollinearity

In [ ]:
def compute_vif(df, label_col):

    # Validation
    validate_input(df, label_col)

    X = df.drop(columns=[label_col])
    X = X.select_dtypes(include=[np.number])

    if X.shape[1] < 2:
        raise ValueError("VIF requires at least two numeric features.")

    vif_values = []

    for i in range(X.shape[1]):
        vif_score = float(variance_inflation_factor(X.values, i))
        vif_values.append({
            "feature": X.columns[i],
            "value": round(vif_score, 2)
        })

    # Sort descending
    vif_values = sorted(vif_values, key=lambda x: x["value"], reverse=True)

    return {
        "vif": vif_values
    }

In [ ]:
compute_vif(df, label_col="Group")

{'vif': [{'feature': 'Vascular_Marker', 'value': 216.07},
  {'feature': 'Cell_Signalling', 'value': 190.91},
  {'feature': 'Tube_formation', 'value': 179.21},
  {'feature': 'Total_ROS', 'value': 155.26},
  {'feature': 'Permeability', 'value': 106.44},
  {'feature': 'In_vivo_recovery', 'value': 80.37},
  {'feature': 'LDL_uptake', 'value': 10.3}]}

### 1). Logistic Regression

#### a) Ridge

In [ ]:
def predict_ridge_logistic(df, label_col, model_path):

    # Validation
    validate_input(df, label_col)

    # Load trained model
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    # Feature validation
    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    # Separate X and y
    X = df[feature_cols]
    y_true = df[label_col]

    # Scaling
    X_scaled = scaler.transform(X)

    # Predictions
    preds = model.predict(X_scaled)
    probs = model.predict_proba(X_scaled)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_true, preds)
    auc_score = roc_auc_score(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    fpr, tpr, _ = roc_curve(y_true, probs)

    # Feature Importance
    coef_series = pd.Series(model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Return result
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": float(auc_score),
        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,
        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        },
        "coefficients": coef_sorted.to_dict()
    }

In [ ]:
predict_ridge_logistic(df, label_col="Group", model_path = "/content/drive/MyDrive/rasp/models/ridge_logistic_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.16598329354312025,
  0.1853378778966807,
  0.80566398468057,
  0.15041494668143973,
  0.8160126093738453,
  0.22800674803161697,
  0.8130968806213781,
  0.8044355800122126,
  0.17016674430817408,
  0.8532796411380738,
  0.18971818573229807,
  0.17850325817988794,
  0.7722335298895411,
  0.8672267158443547],
 'accuracy': 1.0,
 'auc_score': 1.0,
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc_curve': {'fpr': [0.0, 0.0, 0.0, 1.0],


#### b) Lasso

In [ ]:
def predict_lasso_logistic(df, label_col, model_path):

    # Validation
    validate_input(df, label_col)

    # Load trained model
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    # Feature validation
    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    # Separate X and y
    X = df[feature_cols]
    y_true = df[label_col]

    # Scaling
    X_scaled = scaler.transform(X)

    # Predictions
    preds = model.predict(X_scaled)
    probs = model.predict_proba(X_scaled)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_true, preds)
    auc_score = roc_auc_score(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    fpr, tpr, _ = roc_curve(y_true, probs)

    # Feature Importance
    coef_series = pd.Series(model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Return result
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": float(auc_score),
        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,
        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        },
        "coefficients": coef_sorted.to_dict()
    }

In [ ]:
predict_lasso_logistic(df, label_col="Group", model_path="/content/drive/MyDrive/rasp/models/lasso_logistic_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.17056678381681561,
  0.15922684865889872,
  0.8465949832635526,
  0.14470230963961186,
  0.8256124188227777,
  0.16037779572811886,
  0.8567120588453855,
  0.8487107240812296,
  0.17293253324653207,
  0.8538213783543275,
  0.16213412578024228,
  0.12899403715183083,
  0.8169246504143961,
  0.8530595372114944],
 'accuracy': 1.0,
 'auc_score': 1.0,
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc_curve': {'fpr': [0.0, 0.0, 0.0, 1.0

#### c) ElasticNet

In [ ]:
def predict_ElasticNet_logistic(df, label_col, model_path):

    # Validation
    validate_input(df, label_col)

    # Load trained model
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    # Feature validation
    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    # Separate X and y
    X = df[feature_cols]
    y_true = df[label_col]

    # Scaling
    X_scaled = scaler.transform(X)

    # Predictions
    preds = model.predict(X_scaled)
    probs = model.predict_proba(X_scaled)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_true, preds)
    auc_score = roc_auc_score(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    fpr, tpr, _ = roc_curve(y_true, probs)

    # Feature Importance
    coef_series = pd.Series(model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Return result
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": float(auc_score),
        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,
        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        },
        "coefficients": coef_sorted.to_dict()
    }

In [ ]:
predict_ElasticNet_logistic(df, label_col="Group", model_path="/content/drive/MyDrive/rasp/models/elasticnet_logistic_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.20287862038942187,
  0.2215821513835008,
  0.7720618178852382,
  0.18759290444114957,
  0.7812948017343608,
  0.24707154526603972,
  0.7778229878137758,
  0.7718355373366983,
  0.2066638440885038,
  0.8151614721802674,
  0.2261565290163288,
  0.21540159176547127,
  0.740307589791198,
  0.8302554476816068],
 'accuracy': 1.0,
 'auc_score': 1.0,
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc_curve': {'fpr': [0.0, 0.0, 0.0, 1.0],
 

In [ ]:
def logistic_prediction(model, df, label_col="Group"):
    # Ridge Logistic
    if model == "Ridge Logistic Regression":
        model_path = "/content/drive/MyDrive/rasp/models/ridge_logistic_model.pkl"
        prediction_results = predict_ridge_logistic(df=df, label_col=label_col, model_path=model_path)

    # Lasso Logistic
    elif model == "Lasso Logistic Regression":
        model_path = "/content/drive/MyDrive/rasp/models/lasso_logistic_model.pkl"
        prediction_results = predict_lasso_logistic(df=df, label_col=label_col, model_path=model_path)

    # ElasticNet Logistic
    elif model == "ElasticNet Logistic Regression":
        model_path = "/content/drive/MyDrive/rasp/models/elasticnet_logistic_model.pkl"
        prediction_results = predict_ElasticNet_logistic(df=df, label_col=label_col, model_path=model_path)

    else:
        raise ValueError(
            "Invalid model. Choose from: "
            "'Ridge Logistic Regression', "
            "'Lasso Logistic Regression', "
            "'ElasticNet Logistic Regression'"
        )

    return prediction_results

### 2). SVM

In [ ]:
def predict_svm(df, label_col, model_path):
    validate_input(df, label_col)

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    X = df[feature_cols].copy()
    y_true = df[label_col]

    X_scaled = scaler.transform(X)

    preds = model.predict(X_scaled)
    probs = model.predict_proba(X_scaled)[:, 1]

    accuracy = accuracy_score(y_true, preds)

    auc_score = roc_auc_score(y_true, probs)
    fpr, tpr, _ = roc_curve(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": auc_score,

        "confusion_matrix": conf_matrix.tolist(),

        "classification_report": class_report,

        "roc_curve": {
            "fpr": fpr,
            "tpr": tpr
        }
    }

In [ ]:
predict_svm(df, label_col="Group", model_path="/content/drive/MyDrive/rasp/models/svm_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.10272526649681585,
  0.11672684462216602,
  0.8757289875588531,
  0.08695952752068203,
  0.8889260536907472,
  0.11658422744227326,
  0.8881515000527523,
  0.8815139356217191,
  0.10485258672966621,
  0.9210671426788878,
  0.11672272772974225,
  0.10932365314632334,
  0.862798071581367,
  0.9251426886828038],
 'accuracy': 1.0,
 'auc_score': np.float64(1.0),
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc_curve': {'fpr': array([0

## 3) PCA Analysis

### a) Logistic Regression

#### i). Ridge

In [ ]:
def predict_pca_ridge_logistic(df, label_col, model_path):
    validate_input(df, label_col)

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    pca = artifact["pca"]
    feature_cols = artifact["features"]

    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    X = df[feature_cols].copy()
    y_true = df[label_col]

    # Scaling
    X_scaled = scaler.transform(X)

    # PCA transform
    X_pca = pca.transform(X_scaled)

    preds = model.predict(X_pca)
    probs = model.predict_proba(X_pca)[:, 1]

    accuracy = accuracy_score(y_true, preds)
    auc_score = roc_auc_score(y_true, probs)

    fpr, tpr, _ = roc_curve(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    # Component Importance (coefficients)
    coefficients = model.coef_[0]
    component_names = [f"PC{i+1}" for i in range(len(coefficients))]
    component_importance = dict(
        sorted(
            zip(component_names, coefficients),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    )

    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },
        "pca_info": {
            "components_used": int(pca.n_components_)
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": float(auc_score),

        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,

        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        },
        "component_importance": component_importance
    }

In [ ]:
predict_pca_ridge_logistic(df, "Group", "/content/drive/MyDrive/rasp/models/ridge_logistic_pca_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'pca_info': {'components_used': 2},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.16573515392426616,
  0.1854569150299356,
  0.8059742132063268,
  0.15038722153278544,
  0.8162989073828677,
  0.2281364664588286,
  0.8127067379342563,
  0.80453320842568,
  0.1696686871504695,
  0.8530843843532714,
  0.1901113045907234,
  0.17888305644249228,
  0.7711647571589145,
  0.8678292569101261],
 'accuracy': 1.0,
 'auc_score': 1.0,
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc_curv

#### ii). Lasso

In [ ]:
def predict_pca_lasso_logistic(df, label_col, model_path):
    validate_input(df, label_col)

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    pca = artifact["pca"]
    feature_cols = artifact["features"]

    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    X = df[feature_cols].copy()
    y_true = df[label_col]

    # Scaling
    X_scaled = scaler.transform(X)

    # PCA transform
    X_pca = pca.transform(X_scaled)

    preds = model.predict(X_pca)
    probs = model.predict_proba(X_pca)[:, 1]

    accuracy = accuracy_score(y_true, preds)
    auc_score = roc_auc_score(y_true, probs)

    fpr, tpr, _ = roc_curve(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    # Component Importance (coefficients)
    coefficients = model.coef_[0]
    component_names = [f"PC{i+1}" for i in range(len(coefficients))]
    component_importance = dict(
        sorted(
            zip(component_names, coefficients),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    )

    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },
        "pca_info": {
            "components_used": int(pca.n_components_)
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": float(auc_score),

        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,

        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        },
        "component_importance": component_importance
    }

In [ ]:
predict_pca_lasso_logistic(df, "Group", "/content/drive/MyDrive/rasp/models/lasso_logistic_pca_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'pca_info': {'components_used': 2},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.15884302029145952,
  0.17860723673505313,
  0.8081935250498932,
  0.1433125677015634,
  0.8195572187622833,
  0.23851887446374148,
  0.8167762353626605,
  0.8053100184568626,
  0.16478162112533365,
  0.8593499188617061,
  0.18330530832366035,
  0.17136307365488457,
  0.7740245189512834,
  0.8729723279652399],
 'accuracy': 1.0,
 'auc_score': 1.0,
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc

#### iii) ElasticNet

In [ ]:
def predict_ElasticNet_logistic(df, label_col, model_path):

    # Validation
    validate_input(df, label_col)

    # Load trained model
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    pca = artifact["pca"]
    feature_cols = artifact["features"]

    # Feature validation
    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    # Separate X and y
    X = df[feature_cols]
    y_true = df[label_col]

    # Scaling
    X_scaled = scaler.transform(X)

    # PCA transform
    X_pca = pca.transform(X_scaled)

    preds = model.predict(X_pca)
    probs = model.predict_proba(X_pca)[:, 1]

    # Predictions
    preds = model.predict(X_pca)
    probs = model.predict_proba(X_pca)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_true, preds)
    auc_score = roc_auc_score(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    fpr, tpr, _ = roc_curve(y_true, probs)

    # Component Importance
    coefficients = model.coef_[0]
    component_names = [f"PC{i+1}" for i in range(len(coefficients))]

    component_importance = dict(
        sorted(
            zip(component_names, coefficients),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    )

    # Return result
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": float(auc_score),
        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,
        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        },
        "component_importance": component_importance,
    }

In [ ]:
predict_ElasticNet_logistic(df, "Group", "/content/drive/MyDrive/rasp/models/elasticnet_logistic_pca_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.1693586738516977,
  0.18909418241862697,
  0.7967009024241378,
  0.153743100597562,
  0.8080255869888857,
  0.24816023526402115,
  0.8052500340748604,
  0.793834207096616,
  0.17530395547976071,
  0.8480687624254208,
  0.19376489906950542,
  0.18187719195217633,
  0.7628947477336987,
  0.8619380876218081],
 'accuracy': 1.0,
 'auc_score': 1.0,
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0}},
 'roc_curve': {'fpr': [0.0, 0.0, 0.0, 1.0],
 

### b) SVM

In [ ]:
def predict_svm_pca(df, label_col, model_path):
    validate_input(df, label_col)

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    pca = artifact["pca"]
    feature_cols = artifact["features"]

    missing_cols = [c for c in feature_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    X = df[feature_cols].copy()
    y_true = df[label_col]

    # Scale
    X_scaled = scaler.transform(X)

    # PCA transform
    X_pca = pca.transform(X_scaled)

    # Predictions
    preds = model.predict(X_pca)
    probs = model.predict_proba(X_pca)[:, 1]

    accuracy = accuracy_score(y_true, preds)

    auc_score = roc_auc_score(y_true, probs)
    fpr, tpr, _ = roc_curve(y_true, probs)

    conf_matrix = confusion_matrix(y_true, preds)
    class_report = classification_report(y_true, preds, output_dict=True)

    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y_true.value_counts().to_dict()
        },

        "pca_info": {
            "components_used": int(pca.n_components_)
        },

        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),

        "accuracy": float(accuracy),
        "auc_score": auc_score,

        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": class_report,

        "roc_curve": {
            "fpr": fpr.tolist(),
            "tpr": tpr.tolist()
        }
    }

In [ ]:
predict_svm_pca(df, label_col="Group", model_path="/content/drive/MyDrive/rasp/models/svm_pca_model.pkl")

{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'pca_info': {'components_used': 2},
 'predictions': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'probabilities': [0.09851185764466873,
  0.11591819004222033,
  0.8785823960351415,
  0.0860986766895837,
  0.8861535854243873,
  0.1200672318361653,
  0.8809040188735152,
  0.8803794913641572,
  0.09763751761301519,
  0.9125666091888373,
  0.12011798453423601,
  0.1114620828952284,
  0.841431811643958,
  0.9269318710231357],
 'accuracy': 1.0,
 'auc_score': np.float64(1.0),
 'confusion_matrix': [[7, 0], [0, 7]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 7.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 7.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 14.0

In [ ]:
def pca_model_prediction(model, df, label_col="Group"):

    if model == "SVM":
        model_path = "/content/drive/MyDrive/rasp/models/svm_pca_model.pkl"
        prediction_results = predict_svm_pca(df, label_col, model_path)

    elif model == "Ridge Logistic Regression":
        model_path = "/content/drive/MyDrive/rasp/models/ridge_logistic_pca_model.pkl"
        prediction_results = predict_pca_ridge_logistic(df, label_col, model_path)

    elif model == "Lasso Logistic Regression":
        model_path = "/content/drive/MyDrive/rasp/models/lasso_logistic_pca_model.pkl"
        prediction_results = predict_pca_lasso_logistic(df, label_col, model_path)

    # ElasticNet Logistic + PCA
    elif model == "ElasticNet Logistic Regression":
        model_path = "/content/drive/MyDrive/rasp/models/elasticnet_logistic_pca_model.pkl"
        prediction_results = predict_ElasticNet_logistic(df=df, label_col=label_col, model_path=model_path)

    else:
        raise ValueError(
            "Invalid model. Choose from: "
            "'SVM', "
            "'Ridge Logistic Regression', "
            "'Lasso Logistic Regression', "
            "'ElasticNet Logistic Regression'"
        )

    return prediction_results

In [ ]:
def model_prediction(model_type, model, df, label_col="Group"):
    # Logistic models (Ridge, Lasso, ElasticNet)
    if model_type == "logistic":
        return logistic_prediction(model=model, df=df, label_col=label_col)

    # SVM model
    elif model_type == "svm":
        return predict_svm(df=df, label_col=label_col,model_path="/content/drive/MyDrive/rasp/models/svm_model.pkl")

    # PCA models
    elif model_type == "pca":
        return pca_model_prediction(model=model, df=df, label_col=label_col)

    else:
        raise ValueError(
            "Invalid model_type. Choose from: "
            "'logistic', 'svm', 'pca'"
        )

# Clustering Analysis

## 1) K-Means Clustering

In [ ]:
def predict_kmeans(df, label_col, model_path):

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    # Validate features
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    X = df[feature_cols].copy()

    # Scale
    X_scaled = scaler.transform(X)

    # Predict
    labels = model.predict(X_scaled)

    # PCA → 2D (for visualization)
    pca = PCA(n_components=2)
    X_reduced = pca.fit_transform(X_scaled)

    metrics = {}

    # Silhouette (only if >1 cluster)
    if len(set(labels)) > 1:
        metrics["silhouette"] = float(silhouette_score(X_scaled, labels))

    # ARI (only if label_col provided)
    if label_col is not None:
        if label_col not in df.columns:
            raise ValueError(f"{label_col} not found in dataframe")

        y_true = df[label_col]

        if len(y_true) == len(labels):
            metrics["ari"] = float(adjusted_rand_score(y_true, labels))

    return {
        "labels": labels.tolist(),
        "pca_2d": X_reduced.tolist(),
        "metrics": metrics
    }

In [ ]:
predict_kmeans(df, label_col="Group", model_path = "/content/drive/MyDrive/rasp/models/k_means_model.pkl")

{'labels': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'pca_2d': [[2.6585384445536406, -0.37989095200936257],
  [2.433573202879241, -0.3283876752447488],
  [-2.29402013381044, -0.3325305661422246],
  [2.8518179488403597, -0.47344969020258115],
  [-2.4136975647795773, -0.1600044592012271],
  [1.851429024004519, 2.4056794316959578],
  [-2.383882650891135, -0.005398065412061447],
  [-2.264520581824131, -0.5852314414555391],
  [2.588696589628737, 0.036508469917530366],
  [-2.886676855445228, 0.49022232291589735],
  [2.3830137428218383, -0.31321038236309656],
  [2.5136156301051273, -0.4708430766653702],
  [-1.9636489113900484, -0.15581511399437456],
  [-3.0742378846929026, 0.27235119816120096]],
 'metrics': {'silhouette': 0.7546440860425349, 'ari': 1.0}}

## 2). Gaussian Mixture Model (GMM)

In [ ]:
def predict_gmm(df, label_col, model_path):
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    pca = artifact["pca"]
    feature_cols = artifact["features"]

    # Validate features
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    X = df[feature_cols].copy()

    # Enforce order
    X = X[feature_cols]

    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found")

    # Scale
    X_scaled = scaler.transform(X)

    # Predict
    labels = model.predict(X_scaled)

    # Probabilities (GMM strength)
    probs = model.predict_proba(X_scaled)

    # PCA projection
    X_reduced = pca.transform(X_scaled)

    metrics = {}

    # Silhouette (only if >1 cluster)
    if len(set(labels)) > 1:
        metrics["silhouette"] = float(silhouette_score(X_scaled, labels))

    # ARI (only if label_col provided)
    if label_col is not None:
        if label_col not in df.columns:
            raise ValueError(f"{label_col} not found in dataframe")

        y_true = df[label_col]

        if len(y_true) == len(labels):
            metrics["ari"] = float(adjusted_rand_score(y_true, labels))

    return {
        "labels": labels.tolist(),
        "pca_2d": X_reduced.tolist(),
        "probabilities": probs.tolist(),
        "metrics": metrics
    }

In [ ]:
predict_gmm(df, label_col="Group", model_path="/content/drive/MyDrive/rasp/models/GMM_model.pkl")

{'labels': [1, 4, 0, 1, 5, 2, 5, 0, 1, 3, 4, 4, 5, 3],
 'pca_2d': [[2.658538444553638, -0.3798909520093614],
  [2.4335732028792396, -0.3283876752447481],
  [-2.2940201338104376, -0.332530566142225],
  [2.8518179488403574, -0.4734496902025798],
  [-2.4136975647795746, -0.16000445920122797],
  [1.8514290240045175, 2.405679431695956],
  [-2.383882650891133, -0.005398065412062045],
  [-2.2645205818241294, -0.5852314414555391],
  [2.588696589628736, 0.03650846991753089],
  [-2.886676855445227, 0.4902223229158964],
  [2.3830137428218365, -0.3132103823630957],
  [2.513615630105125, -0.4708430766653686],
  [-1.9636489113900475, -0.1558151139943748],
  [-3.0742378846929, 0.2723511981611997]],
 'probabilities': [[0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
  [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
  [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
  [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 1.0, 0.0, 

## 3). Hierarchical Clustering

In [ ]:
def auto_select_k(df, label_col=None, max_k=10):
    df = df.copy()

    # Select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features found")

    X = df[feature_cols].values
    y = df[label_col].values if label_col else None

    # Scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Auto K
    best_k = 2
    best_score = -1

    for k in range(2, min(max_k + 1, len(X_scaled))):
        model = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = model.fit_predict(X_scaled)

        if len(set(labels)) > 1:
            score = silhouette_score(X_scaled, labels)

            if score > best_score:
                best_score = score
                best_k = k

    return X_scaled, y, feature_cols, best_k

In [ ]:
def auto_select_linkage(X_scaled, k):
    linkages = ["ward", "complete", "average"]

    best_linkage = None
    best_score = -1

    for linkage in linkages:
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage, metric="euclidean")

        labels = model.fit_predict(X_scaled)

        if len(set(labels)) > 1:
            score = silhouette_score(X_scaled, labels)

            if score > best_score:
                best_score = score
                best_linkage = linkage

    return best_linkage

In [ ]:
def hierarchical_clustering(df, label_col=None, max_k=10):
    X_scaled, y, feature_cols, best_k = auto_select_k(df, label_col, max_k)

    best_linkage = auto_select_linkage(X_scaled, best_k)

    model = AgglomerativeClustering(n_clusters=best_k, linkage=best_linkage)

    labels = model.fit_predict(X_scaled)

    # Metrics
    metrics = {}

    if len(set(labels)) > 1:
        metrics["silhouette"] = float(silhouette_score(X_scaled, labels))
    else:
        metrics["silhouette"] = None

    if y is not None:
        metrics["ari"] = float(adjusted_rand_score(y, labels))
    else:
        metrics["ari"] = None

    # PCA for visualization
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    # Attach clusters to df
    df_out = df.copy()
    df_out["cluster"] = labels

    return {
        "best_k": best_k,
        "best_linkage": best_linkage,
        "features": feature_cols,
        "metrics": metrics,
        "labels": labels.tolist(),
        "pca_x": X_pca[:, 0].tolist(),
        "pca_y": X_pca[:, 1].tolist(),
    }

In [ ]:
hierarchical_clustering(df, label_col="Group", max_k=10)

{'best_k': 2,
 'best_linkage': 'ward',
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'metrics': {'silhouette': 0.7546440860425349, 'ari': 1.0},
 'labels': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
 'pca_x': [2.6585384445536406,
  2.433573202879241,
  -2.29402013381044,
  2.8518179488403597,
  -2.4136975647795773,
  1.851429024004519,
  -2.383882650891135,
  -2.264520581824131,
  2.588696589628737,
  -2.886676855445228,
  2.3830137428218383,
  2.5136156301051273,
  -1.9636489113900484,
  -3.0742378846929026],
 'pca_y': [-0.37989095200936257,
  -0.3283876752447488,
  -0.3325305661422246,
  -0.47344969020258115,
  -0.1600044592012271,
  2.4056794316959578,
  -0.005398065412061447,
  -0.5852314414555391,
  0.036508469917530366,
  0.49022232291589735,
  -0.31321038236309656,
  -0.4708430766653702,
  -0.15581511399437456,
  0.27235119816120096]}

## 4). DBSCAN

In [ ]:
def fit_dbscan(df, label_col=None, min_samples=5):
    df = df.copy()

    # Select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features found")

    X = df[feature_cols].values
    y = df[label_col].values if label_col else None

    if np.isnan(X).any():
        raise ValueError("NaNs detected — handle before training")

    # Scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Auto eps (merged here)
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors_fit = neighbors.fit(X_scaled)
    distances, _ = neighbors_fit.kneighbors(X_scaled)

    k_distances = np.sort(distances[:, -1])
    eps = np.median(k_distances) * 1.2   # stable heuristic

    # Fit DBSCAN
    model = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean")
    clusters = model.fit_predict(X_scaled)

    clusters = np.array(clusters)

    # Metrics
    metrics = {}

    valid_mask = clusters != -1
    n_valid_clusters = len(set(clusters[valid_mask]))

    if n_valid_clusters > 1:
        metrics["silhouette"] = float(
            silhouette_score(X_scaled[valid_mask], clusters[valid_mask])
        )
    else:
        metrics["silhouette"] = None

    if y is not None:
        metrics["ari"] = float(adjusted_rand_score(y, clusters))
    else:
        metrics["ari"] = None

    return X_scaled, clusters, metrics, eps, feature_cols

In [ ]:
def DBSCAN_predict(df, label_col=None, min_samples=5):

    np.random.seed(42)

    # Step 1+2 merged
    X_scaled, clusters, metrics, eps, feature_cols = fit_dbscan(df, label_col, min_samples)

    # Cluster stats
    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    noise_points = int(np.sum(clusters == -1))
    noise_ratio = noise_points / len(clusters)

    # Safety checks
    if n_clusters < 2:
        raise ValueError("DBSCAN found <2 clusters. Adjust parameters.")

    if noise_ratio > 0.6:
        raise ValueError(f"Too many noise points: {noise_ratio:.2f}")

    # PCA for visualization
    n_components = min(2, X_scaled.shape[1])
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    if X_pca.shape[1] == 1:
        x_coords = X_pca[:, 0].tolist()
        y_coords = [0.0] * len(x_coords)
    else:
        x_coords = X_pca[:, 0].tolist()
        y_coords = X_pca[:, 1].tolist()

    # Attach clusters to df
    df_out = df.copy()
    df_out["cluster"] = clusters

    return {
        "eps": round(float(eps), 4),
        "min_samples": min_samples,
        "n_clusters": int(n_clusters),
        "noise_points": noise_points,
        "noise_ratio": round(float(noise_ratio), 4),
        "features": feature_cols,
        "metrics": metrics,
        "clusters": clusters.tolist(),
        "pca_x": x_coords,
        "pca_y": y_coords,
    }

In [ ]:
DBSCAN_predict(df, label_col="Group", min_samples=5)

{'eps': 1.3921,
 'min_samples': 5,
 'n_clusters': 2,
 'noise_points': 1,
 'noise_ratio': 0.0714,
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'metrics': {'silhouette': 0.8103401601895018, 'ari': 0.865979381443299},
 'clusters': [0, 0, 1, 0, 1, -1, 1, 1, 0, 1, 0, 0, 1, 1],
 'pca_x': [2.6585384445536406,
  2.433573202879241,
  -2.29402013381044,
  2.8518179488403597,
  -2.4136975647795773,
  1.851429024004519,
  -2.383882650891135,
  -2.264520581824131,
  2.588696589628737,
  -2.886676855445228,
  2.3830137428218383,
  2.5136156301051273,
  -1.9636489113900484,
  -3.0742378846929026],
 'pca_y': [-0.37989095200936257,
  -0.3283876752447488,
  -0.3325305661422246,
  -0.47344969020258115,
  -0.1600044592012271,
  2.4056794316959578,
  -0.005398065412061447,
  -0.5852314414555391,
  0.036508469917530366,
  0.49022232291589735,
  -0.31321038236309656,
  -0.4708430766653702,
  -0.1558151139943

In [ ]:
def validate_input(x):
    if not isinstance(x, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if x.empty:
        raise ValueError("Input dataset is empty.")

    if x.isnull().sum().any():
        raise ValueError("Missing values detected in dataset.")

    if not all(np.issubdtype(dtype, np.number) for dtype in x.dtypes):
        raise ValueError("All features must be numeric.")

    if len(x) < 5:
        raise ValueError("Dataset too small for clustering.")

In [ ]:
def clustering_analysis(model, X, y=None, max_clusters=10, user_k=None, **kwargs):
    # Validate Input
    validate_input(X)

    # validate y
    if y is not None:
        if len(y) != len(X):
            raise ValueError("Length of y must match number of samples in X.")


    if model == "KMeans Clustering":
        return predict_kmeans(df, label_col="Group", model_path = "/content/drive/MyDrive/rasp/models/k_means_model.pkl")
    elif model == "GMM Clustering":
        return predict_gmm(df, label_col="Group", model_path="/content/drive/MyDrive/rasp/models/GMM_model.pkl")
    elif model == "Hierarchical Clustering":
        return hierarchical_clustering(df, label_col="Group", max_k=10)
    elif model == "DBSCAN Clustering":
        return DBSCAN_predict(df, label_col="Group", min_samples=5)
    else:
        print("Invalid model")

# Vizualization

## 1) Bar Chart

In [ ]:
def bar_chart_plot(df, label_col, label_map=None, round_digits=4):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric feature columns found.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    df_copy = df.copy()

    # handle label mapping
    if label_map:
        df_copy["Group_label"] = df_copy[label_col].map(label_map)
    else:
        df_copy["Group_label"] = df_copy[label_col].astype(str)

    # compute group means
    means = (
        df_copy.groupby("Group_label")[feature_cols]
        .mean()
        .reset_index()
    )

    # convert to long format
    means_long = means.melt(
        id_vars="Group_label",
        value_vars=feature_cols,
        var_name="Feature",
        value_name="Mean Value"
    )

    # rounding for frontend
    means_long["Mean Value"] = means_long["Mean Value"].round(round_digits)

    return {
        "chart_type": "grouped_bar",
        "data": means_long.to_dict(orient="records")
    }

In [ ]:
bar_chart_plot(df=df, label_col="Group")

{'chart_type': 'grouped_bar',
 'data': [{'Group_label': '0',
   'Feature': 'Permeability',
   'Mean Value': 71.2857},
  {'Group_label': '1', 'Feature': 'Permeability', 'Mean Value': 20.4286},
  {'Group_label': '0', 'Feature': 'LDL_uptake', 'Mean Value': 0.5286},
  {'Group_label': '1', 'Feature': 'LDL_uptake', 'Mean Value': 1.3857},
  {'Group_label': '0', 'Feature': 'Total_ROS', 'Mean Value': 1.9571},
  {'Group_label': '1', 'Feature': 'Total_ROS', 'Mean Value': 0.9429},
  {'Group_label': '0', 'Feature': 'Vascular_Marker', 'Mean Value': 73.4286},
  {'Group_label': '1', 'Feature': 'Vascular_Marker', 'Mean Value': 96.0},
  {'Group_label': '0', 'Feature': 'Cell_Signalling', 'Mean Value': 49.4286},
  {'Group_label': '1', 'Feature': 'Cell_Signalling', 'Mean Value': 86.7143},
  {'Group_label': '0', 'Feature': 'Tube_formation', 'Mean Value': 39.5714},
  {'Group_label': '1', 'Feature': 'Tube_formation', 'Mean Value': 96.0},
  {'Group_label': '0', 'Feature': 'In_vivo_recovery', 'Mean Value': 27.8

## 2). Parallel plot

In [ ]:
def get_parallel_data(df, label_col="Group"):

    # Only feature columns
    feature_cols = df.select_dtypes(include="number").columns.tolist()
    feature_cols.remove(label_col)

    return {
        "chart_type": "parallel",
        "dimensions": feature_cols,
        "data": df[feature_cols].values.tolist(),
        "group": df[label_col].tolist()
    }

In [ ]:
get_parallel_data(df, label_col="Group")

{'chart_type': 'parallel',
 'dimensions': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'data': [[71.0, 0.3, 1.8, 68.0, 43.0, 39.0, 35.0],
  [68.0, 0.34, 1.9, 75.0, 47.0, 35.0, 34.0],
  [20.0, 1.2, 1.0, 100.0, 80.0, 100.0, 70.0],
  [79.0, 0.19, 1.8, 70.0, 43.0, 41.0, 28.0],
  [17.0, 1.3, 0.9, 96.0, 88.0, 92.0, 72.0],
  [69.0, 2.0, 2.2, 74.0, 55.0, 44.0, 19.0],
  [22.0, 1.4, 1.1, 96.0, 85.0, 99.0, 79.0],
  [23.0, 1.1, 0.9, 98.0, 78.0, 98.0, 78.0],
  [70.0, 0.35, 2.1, 69.0, 59.0, 46.0, 22.0],
  [21.0, 1.8, 0.8, 92.0, 95.0, 96.0, 82.0],
  [67.0, 0.3, 1.9, 77.0, 50.0, 40.0, 25.0],
  [75.0, 0.22, 2.0, 81.0, 49.0, 32.0, 32.0],
  [27.0, 1.2, 1.2, 90.0, 89.0, 90.0, 83.0],
  [13.0, 1.7, 0.7, 100.0, 92.0, 97.0, 71.0]],
 'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]}

## 3) PCA Plot

In [ ]:
def pca_plot(df, label_col="Group", n_components=3):
    if n_components not in [2, 3]:
        raise ValueError("n_components must be 2 or 3")

    # Separate features and label
    X = df.drop(columns=[label_col])
    y = df[label_col]

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Explained variance in percentage
    explained = (pca.explained_variance_ratio_ * 100).tolist()

    # Column-wise PCA coordinates for plotting
    points = {
        "pc1": X_pca[:, 0].astype(float).tolist(),
        "pc2": X_pca[:, 1].astype(float).tolist(),
        "group": y.astype(int).tolist()
    }

    if n_components == 3:
        points["pc3"] = X_pca[:, 2].astype(float).tolist()

    # Feature loadings for biplot
    loadings = []
    for i, feature in enumerate(X.columns):
        loading = {
            "feature": feature,
            "pc1": float(pca.components_.T[i, 0]),
            "pc2": float(pca.components_.T[i, 1])
        }
        if n_components == 3:
            loading["pc3"] = float(pca.components_.T[i, 2])
        loadings.append(loading)

    return {
        "chart_type": "pca_3d" if n_components == 3 else "pca_2d",
        "n_components": n_components,
        "explained_variance_percent": explained,
        "points": points,
        "loadings": loadings
    }

    def pca_plot(self, n_components=3):
      if n_components not in [2, 3]:
          raise ValueError("n_components must be 2 or 3")

      df, lc = self.df, self.label_col

      # Separate features and label
      X = df.drop(columns=[lc])
      y = df[lc]

      # Standardize features
      scaler = StandardScaler()
      X_scaled = scaler.fit_transform(X)

      # Apply PCA
      pca = PCA(n_components=n_components)
      X_pca = pca.fit_transform(X_scaled)

      # Explained variance in percentage
      explained = [round(float(v * 100), 2) for v in pca.explained_variance_ratio_]

      groups = y.unique().tolist()

      # Feature loadings table
      loading_rows = []
      for i, feature in enumerate(X.columns):
          row = [feature] + [round(float(pca.components_.T[i, j]), 4) for j in range(n_components)]
          loading_rows.append(row)

      loadings_table = {
          "id": "table-pca-loadings",
          "title": "PCA Feature Loadings",
          "columns": ["Feature"] + [f"PC{j+1}" for j in range(n_components)],
          "rows": loading_rows
      }

      variance_table = {
          "id": "table-pca-variance",
          "title": "PCA Explained Variance",
          "columns": ["Component", "Explained Variance (%)"],
          "rows": [
              [f"PC{i+1}", explained[i]]
              for i in range(n_components)
          ]
      }

      if n_components == 2:
          series = []
          for group in groups:
              mask = y == group
              series.append({
                  "name": str(group),
                  "type": "scatter",
                  "data": [
                      [round(float(X_pca[i, 0]), 4), round(float(X_pca[i, 1]), 4)]
                      for i in range(len(X_pca)) if mask.iloc[i]
                  ],
                  "symbolSize": 8
              })

          chart = {
              "id": "chart-pca-2d",
              "title": "PCA 2D Scatter Plot",
              "fullWidth": True,
              "option": {
                  "tooltip": {"trigger": "item"},
                  "legend": {"data": [str(g) for g in groups]},
                  "xAxis": {
                      "type": "value",
                      "name": f"PC1 ({explained[0]}%)"
                  },
                  "yAxis": {
                      "type": "value",
                      "name": f"PC2 ({explained[1]}%)"
                  },
                  "series": series
              }
          }

      else:
          series = []
          for group in groups:
              mask = y == group
              series.append({
                  "name": str(group),
                  "type": "scatter3D",
                  "data": [
                      [round(float(X_pca[i, 0]), 4), round(float(X_pca[i, 1]), 4), round(float(X_pca[i, 2]), 4)]
                      for i in range(len(X_pca)) if mask.iloc[i]
                  ],
                  "symbolSize": 6
              })

          chart = {
              "id": "chart-pca-3d",
              "title": "PCA 3D Scatter Plot",
              "fullWidth": True,
              "option": {
                  "tooltip": {},
                  "legend": {"data": [str(g) for g in groups]},
                  "xAxis3D": {"type": "value", "name": f"PC1 ({explained[0]}%)"},
                  "yAxis3D": {"type": "value", "name": f"PC2 ({explained[1]}%)"},
                  "zAxis3D": {"type": "value", "name": f"PC3 ({explained[2]}%)"},
                  "grid3D": {"viewControl": {"autoRotate": True}},
                  "series": series
              }
          }

      return {
          "tables": [variance_table, loadings_table],
          "charts": [chart]
      }

In [ ]:
pca_plot(df, label_col="Group", n_components=2)

{'chart_type': 'pca_2d',
 'n_components': 2,
 'explained_variance_percent': [88.55949128638862, 7.553270394983713],
 'points': {'pc1': [2.6585384445536406,
   2.433573202879241,
   -2.29402013381044,
   2.8518179488403597,
   -2.4136975647795773,
   1.851429024004519,
   -2.383882650891135,
   -2.264520581824131,
   2.588696589628737,
   -2.886676855445228,
   2.3830137428218383,
   2.5136156301051273,
   -1.9636489113900484,
   -3.0742378846929026],
  'pc2': [-0.37989095200936257,
   -0.3283876752447488,
   -0.3325305661422246,
   -0.47344969020258115,
   -0.1600044592012271,
   2.4056794316959578,
   -0.005398065412061447,
   -0.5852314414555391,
   0.036508469917530366,
   0.49022232291589735,
   -0.31321038236309656,
   -0.4708430766653702,
   -0.15581511399437456,
   0.27235119816120096],
  'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]},
 'loadings': [{'feature': 'Permeability',
   'pc1': 0.39899903565752076,
   'pc2': 0.06412796620344198},
  {'feature': 'LDL_uptake',
   'pc

## 4) Scatter plot

In [ ]:
def scatter_plot(df, label_col="Group", feature_x=None, feature_y=None, feature_z=None):

    if feature_x is None or feature_y is None:
        raise ValueError("feature_x and feature_y are required")

    if feature_x not in df.columns or feature_y not in df.columns:
        raise ValueError("Selected features not found in dataframe")

    if feature_z and feature_z not in df.columns:
        raise ValueError("feature_z not found in dataframe")

    chart_type = "scatter_3d" if feature_z else "scatter_2d"

    points = {
        feature_x: df[feature_x].astype(float).tolist(),
        feature_y: df[feature_y].astype(float).tolist(),
        "group": df[label_col].astype(int).tolist()
    }

    if feature_z:
        points[feature_z] = df[feature_z].astype(float).tolist()

    return {
        "chart_type": chart_type,
        "features": {
            "x": feature_x,
            "y": feature_y,
            "z": feature_z
        },
        "points": points
    }

In [ ]:
scatter_plot(df, label_col="Group", feature_x="Permeability", feature_y="LDL_uptake", feature_z="Total_ROS")

{'chart_type': 'scatter_3d',
 'features': {'x': 'Permeability', 'y': 'LDL_uptake', 'z': 'Total_ROS'},
 'points': {'Permeability': [71.0,
   68.0,
   20.0,
   79.0,
   17.0,
   69.0,
   22.0,
   23.0,
   70.0,
   21.0,
   67.0,
   75.0,
   27.0,
   13.0],
  'LDL_uptake': [0.3,
   0.34,
   1.2,
   0.19,
   1.3,
   2.0,
   1.4,
   1.1,
   0.35,
   1.8,
   0.3,
   0.22,
   1.2,
   1.7],
  'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
  'Total_ROS': [1.8,
   1.9,
   1.0,
   1.8,
   0.9,
   2.2,
   1.1,
   0.9,
   2.1,
   0.8,
   1.9,
   2.0,
   1.2,
   0.7]}}

In [ ]:
scatter_plot(df, label_col="Group", feature_x="Permeability", feature_y="LDL_uptake", feature_z=None)

{'chart_type': 'scatter_2d',
 'features': {'x': 'Permeability', 'y': 'LDL_uptake', 'z': None},
 'points': {'Permeability': [71.0,
   68.0,
   20.0,
   79.0,
   17.0,
   69.0,
   22.0,
   23.0,
   70.0,
   21.0,
   67.0,
   75.0,
   27.0,
   13.0],
  'LDL_uptake': [0.3,
   0.34,
   1.2,
   0.19,
   1.3,
   2.0,
   1.4,
   1.1,
   0.35,
   1.8,
   0.3,
   0.22,
   1.2,
   1.7],
  'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]}}

## 5) Histogram Plot

In [ ]:
def histogram_plot(df, feature, label_col="Group"):
    if feature not in df.columns:
        raise ValueError(f"{feature} not found in dataframe")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe")

    grouped = (
        df.groupby(label_col)[feature]
        .apply(lambda x: x.astype(float).tolist())
        .to_dict()
    )

    return {
        "chart_type": "histogram",
        "feature": feature,
        "groups": grouped
    }

In [ ]:
histogram_plot(df, feature="Permeability", label_col="Group")

{'chart_type': 'histogram',
 'feature': 'Permeability',
 'groups': {0: [71.0, 68.0, 79.0, 69.0, 70.0, 67.0, 75.0],
  1: [20.0, 17.0, 22.0, 23.0, 21.0, 27.0, 13.0]}}

In [ ]:
def visualization(df, plot_name, label_col=None, **kwargs):

    # basic validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if plot_name == "histogram":
        return histogram_plot(df, feature=kwargs.get("feature"), label_col=label_col)

    elif plot_name == "scatter":
        return scatter_plot(df, label_col=label_col, feature_x=kwargs.get("feature_x"), feature_y=kwargs.get("feature_y"), feature_z=kwargs.get("feature_z"))

    elif plot_name == "pca_plot":
        return pca_plot(df, label_col=label_col, n_components=kwargs.get("n_components", 2))

    elif plot_name == "bar_plot":
        return bar_chart_plot(df, label_col=label_col)

    elif plot_name == "parallel_plot":
        return get_parallel_data(df, label_col=label_col)

    else:
        raise ValueError(f"Invalid plot name: {plot_name}")